# Pipeline YOLO + SAM3 (SAM3-céntrico)

**YOLO localiza (cajas) → SAM3 box-prompt hace las máscaras → green_floor por SAM3 text-prompt.**
SAM3 = centro; YOLO solo acelera. Corre en **GPU del pod** (no en tu laptop).

Ver `context.md` para la explicación completa. Lógica en `pipeline_yolo_sam3.py`.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))
import pipeline_yolo_sam3 as pl

REPO = Path('/workspace/FutBotMX-UAQTeam')
F2 = Path.cwd()
SAM = REPO / 'assets' / 'sam3'
YOLO_PT = F2 / 'best.pt'
VIDEO = REPO / 'data/raw/17Abril/Cámaras/IMG_9871.MOV'   # testing, no visto
print('GPU?', __import__('torch').cuda.is_available())

In [ ]:
# Carga YOLO + SAM3 (tracker para box-prompt, video para green_floor text)
models = pl.load_models(SAM, YOLO_PT, device='cuda')

In [ ]:
# Video 1: solo YOLO (cajas) — rápido
pl.render(VIDEO, F2/'demo_yolo_only.mp4', models, mode='yolo')

In [ ]:
# Video 2: YOLO + SAM3 (cajas -> máscaras box-prompt + green_floor text-prompt)
pl.render(VIDEO, F2/'demo_yolo_sam3.mp4', models, mode='yolo_sam3', green_every=5)

## Notas
- `mode='yolo'`: dibuja solo cajas (demo del localizador rápido).
- `mode='yolo_sam3'`: SAM3 box-prompt convierte cada caja en máscara fina + green_floor.
- `green_every=N`: green_floor (text-prompt, lento) se recalcula cada N frames (región estática).
- Cambia `VIDEO` por cualquier video de testing. Todo en GPU del pod.

**Siguiente:** tracking (IDs), homografía (green_floor), eventos (yellow_zone), LoRA team-aware.